# **PART 1: DATA EXTRACTION AND MANUPULATION**


# **I. Import Library**

In [ ]:
pip install wbgapi

In [ ]:
pip install pandas

In [ ]:
pip install pycountry

In [ ]:
import wbgapi as wb
import pandas as pd
import pycountry
import matplotlib.pyplot as plt

# **II. Extract and Wrangle Data**

## 2.1. Build Function To Extract And Wrangle Data From World Bank API

In [ ]:
def extract_and_wrangle_data(series, countries, year_range):
    """
    Fetches and prepares data from the World Bank API.

    Parameters:
        series (list): List of series/indicators to fetch.
        countries (list): List of country ISO codes.
        year_range (range): Range of years to fetch data for.

    Returns:
        pd.DataFrame: A cleaned and prepared DataFrame.
    """

    # Fetch data from the World Bank API
    df = wb.data.DataFrame(series, countries, year_range).transpose()

    # Automatically map ISO country codes to full country names using pycountry
    def iso3_to_country(iso):
        country = pycountry.countries.get(alpha_3=iso)
        return country.name if country else iso
    df.rename(columns=lambda x: iso3_to_country(x), inplace=True)

    # Sort the columns by the country name for clarity
    df = df.sort_index(axis=1, level=0)

    # Set Time Index
    df.index = pd.to_datetime(df.index.str.lstrip("YR"), format="%Y")
    df.index.name = "year"
    df.index = df.index.year

    return df

- This function downloads and prepares the dataset from the World Bank API. It retrieves the selected indicators for the chosen countries and years, then restructures the data into a usable panel format. Country ISO codes are automatically converted into full country names for clarity.

- The function also creates interaction terms between FDI (as % of GDP) and key conditioning variables such as government effectiveness, electricity access, and school enrollment. These interaction variables are constructed only when the required indicators are available for a country.

- Finally, the dataset is cleaned by sorting columns, converting the time index into numeric years, and returning a structured DataFrame ready for analysis.

## 2.2. Extract and Wrangle Data Into A Dataframe


In [ ]:
df = extract_and_wrangle_data([
    # Variables
    'NY.GDP.PCAP.CD',       #GDP per capita (Dependent variables)
    'BX.KLT.DINV.WD.GD.ZS', #FDI net inflow (% of GDP)
    'BX.KLT.DINV.CD.WD',    #FDI net inflow (USD)
    'NE.GDI.TOTL.KN',       #Gross capital formation (Constant Local Currency Unit)
    'SL.TLF.TOTL.IN',       #Total Labor force
    'NE.TRD.GNFS.ZS',       #Trade Openess (% of GDP)
    'GE.EST',               #Government Effectiveness
    'NY.GDP.DEFL.KD.ZG',    #GDP Deflator
    'SE.TER.ENRR',          #Tertiary Enrollment Rate
    'EG.ELC.ACCS.ZS',       #Access to electricity (% of population)
    ],

    # Countries
    ['IRN', 'SAU',                                 #West Asia
     'TJK',                                       #Central Asia
     'NPL', 'IND', 'PAK', 'BGD',                  #South Asia
     'CHN', 'KHM', 'IDN', 'MYS', 'PHL', 'VNM'],   #East Asia

    # Year Range
    range(2000, 2024))

In [ ]:
df.rename(columns={
    'NY.GDP.PCAP.CD':       'gdp/cap',
    'BX.KLT.DINV.WD.GD.ZS': 'fdi_net/gdp',
    'BX.KLT.DINV.CD.WD':    'fdi_net_inflow',
    'NE.GDI.TOTL.KN':       'gcf',
    'SL.TLF.TOTL.IN':       'labour',
    'NE.TRD.GNFS.ZS':       'trade',
    'GE.EST':               'gov_effect',
    'NY.GDP.DEFL.KD.ZG':    'gdp_deflat',
    'SE.TER.ENRR':          'school_enroll',
    'EG.ELC.ACCS.ZS':       'elec_access',
}, inplace=True)

In [ ]:
df

economy     Bangladesh                                                   \
series  fdi_net_inflow fdi_net/gdp elec_access gov_effect           gcf   
year                                                                      
2000      2.803846e+08    0.525362        32.0  -0.607652  1.717083e+12   
2001      7.852704e+07    0.145444        35.0        NaN  1.853329e+12   
2002      5.230493e+07    0.095579        37.8  -0.726736  1.990848e+12   
2003      2.682852e+08    0.445961        40.5  -0.838939  2.143294e+12   
2004      4.489054e+08    0.689472        40.6  -0.932883  2.319228e+12   
2005      8.133220e+08    1.170652        44.2  -0.918954  2.545669e+12   
2006      4.565232e+08    0.635864        50.5  -0.838699  2.797384e+12   
2007      6.510297e+08    0.817757        46.5  -0.707533  2.997309e+12   
2008      1.328423e+09    1.449658        54.3  -0.738396  3.291498e+12   
2009      9.012866e+08    0.879517        57.1  -0.802756  3.534679e+12   
2010      1.232258e+09    1.068968        55.3  -0.753690  3.837411e+12   
2011      1.264725e+09    0.983399        59.6  -0.772121  4.204303e+12   
2012      1.584403e+09    1.188504        65.5  -0.807726  4.648603e+12   
2013      2.602962e+09    1.735320        61.5  -0.797781  4.897984e+12   
2014      2.539191e+09    1.468703        62.4  -0.775071  5.380682e+12   
2015      2.831153e+09    1.450782        74.0  -0.752010  5.763723e+12   
2016      2.332725e+09    0.879528        75.9  -0.705848  6.277234e+12   
2017      1.810396e+09    0.616342        88.0  -0.752013  6.801984e+12   
2018      2.421626e+09    0.753549        86.9  -0.762284  7.627262e+12   
2019      1.908045e+09    0.543244        92.2  -0.753540  8.152039e+12   
2020      1.525312e+09    0.407860        96.2  -0.779491  8.473708e+12   
2021      1.723856e+09    0.414118        99.0  -0.653352  9.159297e+12   
2022      1.601074e+09    0.347960        99.4  -0.763355  1.022666e+13   
2023      1.484418e+09    0.339361        99.5  -0.697334  1.045277e+13   

economy                                                               ...  \
series       trade gdp_deflat      gdp/cap school_enroll      labour  ...   
year                                                                  ...   
2000     29.321714   3.446659   396.670730      5.251660  49048139.0  ...   
2001     32.098017   3.261160   394.656410      6.172440  50230500.0  ...   
2002     28.967381   3.892867   393.886422      5.915600  51244757.0  ...   
2003     27.657885   5.815817   426.748808      5.991280  52208969.0  ...   
2004     26.858234   4.562136   455.614017      5.556670  53140734.0  ...   
2005     34.396935   4.586361   480.085851      6.134840  54049068.0  ...   
2006     38.111924   5.875936   490.388027      7.070400  54908227.0  ...   
2007     39.942383   6.471260   537.955493      7.663050  55892527.0  ...   
2008     42.620914   7.860966   613.062041      8.622660  56846810.0  ...   
2009     40.092796   6.764355   679.211477     10.493930  57807404.0  ...   
2010     37.802843   7.144663   757.385280           NaN  58796583.0  ...   
2011     47.420850   7.859451   837.336945     13.266325  59281113.0  ...   
2012     48.110923   8.164574   859.680536     13.421160  59869794.0  ...   
2013     46.296403   7.174953   958.262990     13.490058  60484221.0  ...   
2014     44.514080   5.668789  1094.461997     13.577739  61204476.0  ...   
2015     42.085996   5.872777  1224.386477     15.573993  61920877.0  ...   
2016     31.334150  27.850738  1649.283809     17.398521  62629015.0  ...   
2017     29.999731   5.047598  1811.082217     18.517560  66147765.0  ...   
2018     32.514632   5.805539  1965.243727     19.571049  67683856.0  ...   
2019     31.578054   3.658138  2129.798970     22.709464  69165404.0  ...   
2020     26.271447   3.841063  2248.850788     22.692302  70314500.0  ...   
2021     27.724005   4.121175  2482.849178     23.900682  72637932.0  ...   
2022     33.779967   5.049022  2716.485928     23

The dataset is extracted from the WorldBank API into a multi-country panel dataset of 24 years and 16 Asian countries.

## 2.3. Handle Missing Data With Interpolation

***Check for missing data***

In [ ]:
# Count NaNs in every column
missing_counts = df.isna().sum()

# Sort by count descending
missing_counts = missing_counts.sort_values(ascending=False)

# Inspect
print(missing_counts.head(25))

economy                    series        
Cambodia                   school_enroll     4
Pakistan                   school_enroll     3
Viet Nam                   school_enroll     3
Philippines                school_enroll     2
Tajikistan                 school_enroll     2
Nepal                      school_enroll     1
                           gcf               1
Bangladesh                 gov_effect        1
Cambodia                   gov_effect        1
Bangladesh                 school_enroll     1
Iran, Islamic Republic of  gov_effect        1
                           school_enroll     1
Nepal                      gov_effect        1
Pakistan                   gov_effect        1
China                      gov_effect        1
India                      gov_effect        1
Malaysia                   gov_effect        1
Indonesia                  gov_effect        1
Tajikistan                 gov_effect        1
Viet Nam                   gov_effect        1
Saudi Arabia      

***Conduct Interplation Methods***

**Interpolation** works by inferring the missing values through patterns or trends in the data. In this study, **linear interpolation** is used, where a missing value is filled in by connecting the two known values before and after it with a straight line. This assumes that the change between two points is constant over time.

Using interpolation helps preserve the structure and continuity of the dataset without distorting trends, making it particularly suitable for economic variables

In [ ]:
# Handle missing values
df.interpolate(method='linear', limit_direction='both', inplace=True)

In [ ]:
# Count NaNs in every column
missing_counts = df.isna().sum()

# Sort by count descending
missing_counts = missing_counts.sort_values(ascending=False)

# Inspect
print(missing_counts.head(20))

economy     series        
Bangladesh  fdi_net_inflow    0
            fdi_net/gdp       0
            elec_access       0
            gov_effect        0
            gcf               0
            trade             0
            gdp_deflat        0
            gdp/cap           0
            school_enroll     0
            labour            0
Cambodia    fdi_net_inflow    0
            fdi_net/gdp       0
            elec_access       0
            gov_effect        0
            gcf               0
            trade             0
            gdp_deflat        0
            gdp/cap           0
            school_enroll     0
            labour            0
dtype: int64


After the interpolation, there is no more missing values for all variables.

## 2.4. Check Data Type

In [ ]:
df.dtypes.value_counts()

,count
float64,130


All data values are in numerous type, therefore it is ready for the next step.

# **III. Data Description and Sample Selection**

## 3.1. Country Selection


In [ ]:
list(df.columns.get_level_values(0).unique())

['Bangladesh',
 'Cambodia',
 'China',
 'India',
 'Indonesia',
 'Iran, Islamic Republic of',
 'Malaysia',
 'Nepal',
 'Pakistan',
 'Philippines',
 'Saudi Arabia',
 'Tajikistan',
 'Viet Nam']

In this analysis, I have chosen **13 Asian countries** situated in 4 different regions of Asia (West Asia, Central Asia, South Asia, and East Asia) that have the most significant amount of FDI inflow (Above 3 % of GDP). I have excluded some countries in this Asian developing economies, namely Uzbekistan, Thailand, Jordan, Sri Lanka, Kazakhstan, Kyrgyzstan, Turkmenistan, Mongolia, and Myanmar due to their significant missing data on the variables of interest (all of the missing data accounted for more than 30% of the total observation for each variable for each country)

## 3.2. Variable Selection

In [ ]:
variables = list(df.columns.get_level_values(1).unique())
variables

['fdi_net_inflow',
 'fdi_net/gdp',
 'elec_access',
 'gov_effect',
 'gcf',
 'trade',
 'gdp_deflat',
 'gdp/cap',
 'school_enroll',
 'labour']

For the variables, I used GDP per cap as the dependent variable, while the rest will be the independent variables to explain the impact of FDI on economic growth. The reasons for these choices will be compiled into a table below:

| Variable | Type | Definition |
|:---------|:-----------------|:----------------------------------------------|
| **gdp/cap** | **Dependent variable** | Measure economic growth metric (logged real GDP per capita) |
| **fdi_net_inflow** | Independent variable | Measures the actual addition of foreign capital to the domestic economy |
| **fdi/gdp** | For reference only | Measure scale of inward foreign direct investment capital |
| **elec_access** | Independent variable | Meausure infrastructure quality => Better power access boosts absorptive capacity |
| **gov_effect** | Independent variable | Measure institutional quality => Efficient government increases FDI inflow management effectiveness |
| **gcf** | Independent variable | Measure domestic physical-capital accumulation (Solow framework) => Allow the model to distinguish between foreign and domestic sources of capital|
| **trade** | Independent variable | Measure trade openness => Trade can complement or substitute for FDI’s spill-overs |
| **school_enroll** | Independent variable | Measure Human-capital stock at 	tertiary enrolment level => Educated workers learn more from foreign firms and new technology transfer |
| **labour** | Independent variable | Measure labour-input / population-size control |

## 3.3. Time Period

In [ ]:
df.index

Index([2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011,
       2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023],
      dtype='int32', name='year')

The reason for choosing this 23-year period in the research is due to its data consistency and multiple economic cycles coverage of Asian countries:
- **2000s**: FDI boom and Asia's after crisis
- **2008-2009**: global financial crisis and recovery
- **2010-2014**: Commodity-price super-cycle
- **2020-2023**: COVID-19 shock and rebound

Moreover, after 2000, most Asian economies transformed FDI policies, joined the WTO (China in 2001, Vietnam in 2007), or signed major regional FTAs during this period, attracting more significant FDI inflow

# **IV. Reframe and Save Data**

## 4.1. Reframe Dataset to Panel Data

In [ ]:
df = (
    df
    .stack(level=0)                 # move country level into rows
    .reset_index()                  # index → columns
    .rename(columns={'economy': 'country'})
)

# Ensure column order: country, year,...
front_cols = ['country', 'year']
other_cols = [c for c in df.columns if c not in front_cols]

df = df[front_cols + other_cols]

# Sort
df = df.sort_values(['country', 'year']).reset_index(drop=True)

/tmp/ipython-input-8177/1187588256.py:3: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(level=0)                 # move country level into rows


In [ ]:
# Get iso code for data analysis:

# Build mapping
def country_to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return np.nan

iso_map = {
    c: country_to_iso3(c)
    for c in df['country'].unique()
}

# Map to dataframe
df['iso_alpha3'] = df['country'].map(iso_map)

# Reorder columns:
cols = df.columns.tolist()
cols.insert(1, cols.pop(cols.index('iso_alpha3')))
df = df[cols]


In [ ]:
df

series,country,iso_alpha3,year,fdi_net_inflow,fdi_net/gdp,elec_access,gov_effect,gcf,trade,gdp_deflat,gdp/cap,school_enroll,labour
0,Bangladesh,BGD,2000,2.803846e+08,0.525362,32.0,-0.607652,1.717083e+12,29.321714,3.446659,396.670730,5.251660,49048139.0
1,Bangladesh,BGD,2001,7.852704e+07,0.145444,35.0,-0.667194,1.853329e+12,32.098017,3.261160,394.656410,6.172440,50230500.0
2,Bangladesh,BGD,2002,5.230493e+07,0.095579,37.8,-0.726736,1.990848e+12,28.967381,3.892867,393.886422,5.915600,51244757.0
3,Bangladesh,BGD,2003,2.682852e+08,0.445961,40.5,-0.838939,2.143294e+12,27.657885,5.815817,426.748808,5.991280,52208969.0
4,Bangladesh,BGD,2004,4.489054e+08,0.689472,40.6,-0.932883,2.319228e+12,26.858234,4.562136,455.614017,5.556670,53140734.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,Viet Nam,VNM,2019,1.612000e+10,4.821075,99.4,0.027881,1.777370e+15,164.704215,2.423227,3440.900254,30.224119,56180127.0
308,Viet Nam,VNM,2020,1.580000e+10,4.558362,99.8,0.193558,1.850454e+15,163.245863,1.467478,3534.039535,36.089808,55149609.0
309,Viet Nam,VNM,2021,1.566000e+10,4.273146,100.0,0.243436,1.911196e+15,186.675833,2.880768,3704.193559,41.955497,55299911.0
310,Viet Nam,VNM,2022,1.790000e+10,4.329473,100.0,0.174635,2.019205e+15,183.153619,4.442833,4147.697772,44.753685,56475416.0


## 4.2. Save Dataset to Google Drive

In [ ]:
# Mount Google account
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Save df as csv file
df.to_csv('/content/drive/MyDrive/2. Data Projects/1. FDI x Econ Growth/FDI_EconGrowth_Data.csv', index=False)